# LCEL 고급문법으로 Runnable 활용하기

## 1. RunnablePassthrough

`RunnablePassthrough`는 데이터를 받아서 그대로 전달하는 역할

이 클래스는 invoke() 메서드를 통해 받은 데이터를 그대로 돌려줌

겉보기는 '그냥 전달만 한다면 왜 필요한가?'라는 생각이 들 수 있음

하지만 RunnablePassthrough는 마치 메시지를 전달하는 중간 다리처럼 파이프라인 안에서 자리를 잡아 주는 역할

덕분에 다음 단계로 데이터를 잘 넘겨 줄 수 있음

또한 RunnablePassthrough는 단순히 입력을 그대로 넘길 수도 있지만, 추가적인 키를 덧붙여서 함께 전달할 수도 있음

이 기능은 RunnblePassthrough처럼 여러 작업을 동시에 실행하는 도구와 같이 사용할 때 유용

이 경우 전달된 데이터를 맵 구조의 새로운 키에 담아 배분하는 데 활용

In [1]:
from langchain_teddynote import logging
from dotenv import load_dotenv

load_dotenv()
logging.langsmith("test0914")

LangSmith 추적을 시작합니다.
[프로젝트명]
test0914


RunnableParallel과 RunnablePassthrough를 가져옴

RunnableParallel은 병렬 처리를 하는데, 입력 값을 여러 군데에서 받아서 이것으 적절한 키에 할당해서 전달

즉 해당 키에 데이터를 전달 받고 passed, extra, modified처럼 임의로 지정한 속성에 딕셔너리 형태로 값을 담아 전달하는 역할

- `passed`: RunnablePassthrough 인스턴스를 할당하여 입려을 그대로 반환하도록 설정. 단순히 {'num': 1}을 전달
- `extra`: RunnablePassthrough.assign() 메서드를 사용하여 입력의 "num" 값에 3을 곱한 결과를 "mult" 키에 할당하는 작업을 정의. 이 경우 extra는 원래 값에 mult 키가 추가된 {'num': 1, 'mult': 3}로 설정
- `modified`: 람다 함수를 사용하여 입력의 "num" 값에 1을 더하는 작업을 정의. 그 결과 modified 키의 값은 2가 됨

In [2]:
# !pip install langchain_core

In [3]:
from langchain_core.runnables import RunnableParallel, RunnablePassthrough

runnable = RunnableParallel(
    # 전달된 입력을 그대로 반환하는 Runnable을 설정
    passed = RunnablePassthrough(),
    # 입력의 "num" 값에 3을 곱한 결과를 반환하는 Runnable을 설정
    extra = RunnablePassthrough.assign(mult=lambda x: x["num"] * 3),
    # 입력의 "num" 값에 1을 더한 결과를 반환하는 Runnable을 설정
    modified = lambda x: x["num"] + 1
)

In [4]:
# runnable 객체에 invoke() 메서드를 호출하여 {"num": 1}을 입력해 병렬 작업을 실행
runnable.invoke({"num": 1})     # Runnable을 실행

{'passed': {'num': 1}, 'extra': {'num': 1, 'mult': 3}, 'modified': 2}

In [6]:

# assign() 메서드는 전달 받은 키('num': 1)와 새로 지정하는 키('mult': 3)를 반환해 줌.
# r 변수에 넣은 값을 출력해 보면 RunnablePassthrough에 전달받은 'num': 1을 그대로 반환했고, 새로 추가한 mult라는 키에 람다 함수로 num값('num': 1)과 3을 곱한 값을 계산해 'mult': 3을 반환 했음을 알 수 있음
r = RunnablePassthrough.assign(mult=lambda x: x["num"] * 3)
r.invoke({"num": 1})

{'num': 1, 'mult': 3}

In [ ]:
# !pip install faiss-cpu

In [10]:
# 이번에는 리트리버 예제에서 RunnablePassthrough를 어떻게 사용하는지 살펴봄
# 4 개의 문장으로 벡터 스토어 예시를 만들어 봤으며, 리트리버를 가져오고 템플릿을 설정
# 리트리버 체인을 생성하는 부분을 보면 RunnablePassthrough를 써서 사용자가 입력한 질문이 retrieval_chain의 retriever와 RunnablePassthrough()에 동시에 전달
# 즉. 프롬프트 입력 변수 question에는 질문 텍스트가 그대로 담기고, context에는 질문에 대한 결과가 들어감
from langchain_community.vectorstores import FAISS
from langchain_core.output_parsers import StrOutputParser
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.runnables import RunnablePassthrough
from langchain_openai import ChatOpenAI, OpenAIEmbeddings

vectorstore = FAISS.from_texts(     # 벡터 스토어를 생성
    [
        "John used to work at Google.",
        "Gordon and John worked at the same company.",
        "John is a software engineer.",
        "Gordon is a web designer.",
    ],
    embedding = OpenAIEmbeddings(),
)
retriever = vectorstore.as_retriever()  # 벡터 스토어를 리트리버로 사용
# 템플릿 정의
template = """Answer the question based only on the following context:
{context}

Question: {question}
"""
prompt = ChatPromptTemplate.from_template(template)
model = ChatOpenAI(model="gpt-4o-mini")

def format_docs(docs):  # 문서를 포매팅
    return "\n".join([doc.page_content for doc in docs])

retrieval_chain = (
    {"context": retriever | format_docs, "question": RunnablePassthrough()}
    | prompt
    | model
    | StrOutputParser()
)

retrieval_chain.invoke("What is John's job?")

"John's job is a software engineer."

## 2. Runnable 구조 검토하기

LCEL로 Runnable 객체를 생성한 후에는 종종 이를 검사하여 어떤 일이 일어나고 있는지 더 잘 파악하고 싶을 것

이때 LangSmith를 활용해도 되지만, 여기서는 Runnable 체인의 구조를 그래프로 출력해서 확인

In [ ]:
# 리트리버를 생성하고 그래프를 그리기 위한 라이브러리 설치 명령어
# !pip install -qU faiss-cpu tiktoken
# !pip install -qU grandalf

In [14]:
from langchain_classic.prompts import ChatPromptTemplate
from langchain_community.vectorstores import FAISS
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnableLambda, RunnablePassthrough
from langchain_openai import ChatOpenAI, OpenAIEmbeddings
from langchain_teddynote import logging
from dotenv import load_dotenv

load_dotenv()
logging.langsmith("test0914")

vectorstore = FAISS.from_texts(
    # 텍스트 데이터로부터 FAISS 벡터 스토어를 생성
    ["John is an software engineer who loves programming."],
    embedding=OpenAIEmbeddings()
)

# 벡터 스토어를 기반으로 retriever를 생성 
retriever = vectorstore.as_retriever()

template = """Answer the question based only on the following context:
{context}
 
Question: {question}"""

# 템플릿을 기반으로 ChatPromptTemplate을 생성
prompt = ChatPromptTemplate.from_template(
    template
)

model = ChatOpenAI(model="gpt-4o-mini")     # ChatOpenAI 모델을 초기화

chain = (   # chain을 생성
    # 검색 컨텍스트와 질문을 지정
    {"context": retriever, "question": RunnablePassthrough()}
    | prompt                # 프롬프트 생성
    | model                 # 언어 모델 실행
    | StrOutputParser()     # 출력 결과를 문자열로 파싱
)

LangSmith 추적을 시작합니다.
[프로젝트명]
test0914


In [15]:
# chain.get_graph() 메서드는 체인의 실행 그래프를 반환함
# 이 메서드는 체인의 각 노드와 노드 간의 연결을 나타내는 그래프 객체를 반환
chain.get_graph().nodes # 체인의 그래프에서 노드 가져오기

{'c4e7dcc449644c208410c7d394c98f06': Node(id='c4e7dcc449644c208410c7d394c98f06', name='Parallel<context,question>Input', data=<class 'langchain_core.runnables.base.RunnableParallel<context,question>Input'>, metadata=None),
 'f6f60b3d7ea040d48b7d7ee4d085c72f': Node(id='f6f60b3d7ea040d48b7d7ee4d085c72f', name='Parallel<context,question>Output', data=<class 'langchain_core.utils.pydantic.RunnableParallel<context,question>Output'>, metadata=None),
 '52a71cb4169d4d2c9efbbb71eb001ed8': Node(id='52a71cb4169d4d2c9efbbb71eb001ed8', name='VectorStoreRetriever', data=VectorStoreRetriever(tags=['FAISS', 'OpenAIEmbeddings'], vectorstore=<langchain_community.vectorstores.faiss.FAISS object at 0x0000027996590050>, search_kwargs={}), metadata=None),
 'e66876aca2bb493cbbdbff1b466ae0a5': Node(id='e66876aca2bb493cbbdbff1b466ae0a5', name='Passthrough', data=RunnablePassthrough(), metadata=None),
 '98d14162a5d04d42ab07cb694f365d98': Node(id='98d14162a5d04d42ab07cb694f365d98', name='ChatPromptTemplate', dat

In [ ]:
chain.get_graph().edges # 체인의 그래프에서 edge 가져오기

[Edge(source='e981b0dec203442c89d31b79ce99235d', target='ef82bc0036254129ace29c6786f1f9b6', data=None, conditional=False),
 Edge(source='ef82bc0036254129ace29c6786f1f9b6', target='be049ff1aefb4dc7b326cc583c20a16d', data=None, conditional=False),
 Edge(source='e981b0dec203442c89d31b79ce99235d', target='38fb39d46a894d7e933d1ca7311755a5', data=None, conditional=False),
 Edge(source='38fb39d46a894d7e933d1ca7311755a5', target='be049ff1aefb4dc7b326cc583c20a16d', data=None, conditional=False),
 Edge(source='be049ff1aefb4dc7b326cc583c20a16d', target='fc97208b658042368d5cb61fd267fff7', data=None, conditional=False),
 Edge(source='fc97208b658042368d5cb61fd267fff7', target='95c4465be5684803ace626a684e09c47', data=None, conditional=False),
 Edge(source='9edb4e6d01b848d6a041138603f1feb3', target='b1e365b3d8a64d25a1c9d1ce95d2bea4', data=None, conditional=False),
 Edge(source='95c4465be5684803ace626a684e09c47', target='9edb4e6d01b848d6a041138603f1feb3', data=None, conditional=False)]

그래프의 노드(node)는 체인의 각 단계를 나타내며, 에지(edge)는 단계 간의 데이터 흐름을 나타냄

In [18]:
# 체인의 node와 edge를 ASCII 형식의 그래프로 출력
# 맨 처음 node를 보면 context와 question에 하나의 입력 값이 들어오면 리트리버와 RunnablePassthrough에 동시에 전달됨
chain.get_graph().print_ascii()

           +---------------------------------+         
           | Parallel<context,question>Input |         
           +---------------------------------+         
                    **               **                
                 ***                   ***             
               **                         **           
+----------------------+              +-------------+  
| VectorStoreRetriever |              | Passthrough |  
+----------------------+              +-------------+  
                    **               **                
                      ***         ***                  
                         **     **                     
           +----------------------------------+        
           | Parallel<context,question>Output |        
           +----------------------------------+        
                             *                         
                             *                         
                             *                  

In [19]:
# chain.get_prompts() 메서드는 체인에서 사용되는 prompt 객체의 리스트를 반환함
chain.get_prompts()

[ChatPromptTemplate(input_variables=['context', 'question'], input_types={}, partial_variables={}, messages=[HumanMessagePromptTemplate(prompt=PromptTemplate(input_variables=['context', 'question'], input_types={}, partial_variables={}, template='Answer the question based only on the following context:\n{context}\n\nQuestion: {question}'), additional_kwargs={})])]